In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agentic/long-running-agents-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 01 · The Durable Agent Loop — worked example

**What you'll see:** an agent loop that survives a process crash *after* a side effect and still charges the card exactly once.

The whole pattern in one line: **wake → do one step → checkpoint → sleep**. Nothing waits in memory; a *dispatcher* (Cloud Tasks in prod) delivers the next wake-up.

```mermaid
flowchart LR
  T[Cloud Tasks<br/>named task] --> H[POST /internal/tasks/step]
  H --> L[acquire lease]
  L --> P{pending STARTED<br/>step?}
  P -- yes --> X[execute idempotently]
  P -- no --> M[LLM decides]
  M -- final --> S[SUCCEEDED]
  M -- tool --> I[journal intent<br/>checkpoint #1]
  I --> X
  X --> C[checkpoint #2]
  C --> N[enqueue next<br/>named task]
```

Everything below runs offline: `ScriptedLLM` stands in for Gemini, `InMemoryRunStore` for Firestore, `InMemoryDispatcher` for Cloud Tasks. The GCP implementations are in `src/lragents/core/firestore_store.py` and `transport.py`.

In [1]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))          # repo root when run from notebooks/
sys.path[:0] = [os.path.join(ROOT, "src"), os.path.join(ROOT, "notebooks")]

def show_journal(run):
    print(f"run {run.run_id}  status={run.status.value}  version={run.version}  steps={run.usage.steps}  tokens={run.usage.tokens}  cost=${run.usage.cost_usd:.4f}")
    for s in run.journal:
        out = json.dumps(s.output, default=str)[:70] if s.output is not None else (s.error or "")
        print(f"  [{s.index}] {s.kind.value:<6} {s.status.value:<7} {s.name:<18} key={s.idempotency_key or '-':<20} {out}")

In [2]:
from lragents.core import *
from lragents.patterns import DurableAgentLoop

gateway = PaymentGateway()                       # a downstream API that honours Idempotency-Key

def lookup_price(args, ctx):
    return {"sku": args["sku"], "price": 42.0}

def charge_card(args, ctx):                      # ctx.idempotency_key is stable across retries
    return gateway.charge(float(args["amount"]), idempotency_key=ctx.idempotency_key)

tools = ToolRegistry([
    Tool("lookup_price", "Look up a SKU price", lookup_price),
    Tool("charge_card",  "Charge the card",     charge_card),
])

SCRIPT = [Decision.call("lookup_price", sku="ABC"),
          Decision.call("charge_card", amount=42.0),
          Decision.final("Charged 42.00 for ABC")]

def build(script=SCRIPT, faults=None, clock=None, gateway=gateway):
    clock = clock or FakeClock()
    dispatcher = InMemoryDispatcher(clock=clock)
    loop = DurableAgentLoop(store=InMemoryRunStore(clock=clock), llm=ScriptedLLM(list(script)), tools=tools,
                            dispatcher=dispatcher, idempotency=InMemoryIdempotencyStore(),
                            price=PriceCard(input_per_m=0.5, output_per_m=3.0), clock=clock, faults=faults)
    return loop, dispatcher, clock

## 1. Happy path
Each `deliver_one` is one Cloud Tasks delivery → one HTTP request → one step.

In [3]:
loop, dispatcher, clock = build()
run = loop.start("Buy SKU ABC")
while dispatcher.deliver_one(loop.handle):
    r = loop.store.get(run.run_id)
    print(f"after wake-up: status={r.status.value:<10} journal={len(r.journal)} pending_tasks={dispatcher.pending()}")
show_journal(loop.store.get(run.run_id))
print("charges:", [(c.key, c.amount) for c in gateway.charges])

after wake-up: status=RUNNING    journal=2 pending_tasks=1
after wake-up: status=RUNNING    journal=4 pending_tasks=1
after wake-up: status=SUCCEEDED  journal=5 pending_tasks=0
run run_b98b2618e70d  status=SUCCEEDED  version=12  steps=5  tokens=1060  cost=$0.0009
  [0] llm    done    decide             key=-                    {"kind": "tool_call", "tool": "lookup_price", "args": {"sku": "ABC"}, 
  [1] tool   done    lookup_price       key=run_b98b2618e70d:1   {"result": {"sku": "ABC", "price": 42.0}, "replayed": false}
  [2] llm    done    decide             key=-                    {"kind": "tool_call", "tool": "charge_card", "args": {"amount": 42.0},
  [3] tool   done    charge_card        key=run_b98b2618e70d:3   {"result": {"charge_id": "run_b98b2618e70d:3", "amount": 42.0, "replay
  [4] llm    done    decide             key=-                    {"kind": "final", "tool": null, "args": {}, "text": "Charged 42.00 for
charges: [('run_b98b2618e70d:3', 42.0)]


Read the journal: every LLM decision is recorded *before* the tool runs, every tool step carries an idempotency key `run:index`, and `usage` accumulates tokens/cost so the budget guard has something to check.

## 2. Crash after the side effect
The dangerous window: the card **has been charged** but checkpoint #2 has not been written. We kill the process there.

In [4]:
gateway2 = PaymentGateway()
faults = FaultInjector()
loop, dispatcher, clock = build(faults=faults, gateway=gateway2)
tools.get("charge_card").fn = lambda a, c: gateway2.charge(float(a["amount"]), idempotency_key=c.idempotency_key)

run = loop.start("Buy SKU ABC")
dispatcher.deliver_one(loop.handle)                 # step 0: lookup
faults.crash_once_at("after_side_effect")           # next tool call dies after acting, before saving
env = dispatcher.queue.pop(0)
try:
    loop.handle(env)
except SimulatedCrash as e:
    print("💥", e)

mid = loop.store.get(run.run_id)
print("charges so far:", len(gateway2.charges))
print("pending step  :", mid.pending_step().name, mid.pending_step().status.value, mid.pending_step().idempotency_key)
print("lease         :", mid.lease)

💥 simulated crash at after_side_effect
charges so far: 1
pending step  : charge_card started run_531eddcd76e4:3
lease         : Lease(owner='worker-1', expires_at=1700000060.0)


The store says: *intent recorded, not finished*. The lease is still held by the dead worker — a real crash never releases it. Cloud Tasks retries the delivery; the retry must **wait for the lease to expire** (TTL), then take the recovery path.

In [5]:
retry_worker = DurableAgentLoop(store=loop.store, llm=loop.llm, tools=tools, dispatcher=dispatcher,
                                idempotency=loop.idem, worker_id="worker-2", clock=clock, price=loop.price)
try:
    retry_worker.handle(env)
except LeaseHeld as e:
    print("retry refused:", e)

clock.advance(61)                                    # lease TTL is 60s
retry_worker.handle(env)                             # recovery: re-executes the SAME intent under the SAME key
dispatcher.drain(retry_worker.handle)
final = loop.store.get(run.run_id)
show_journal(final)
print("charges:", len(gateway2.charges), "| recoveries:", final.state.get("recoveries"), "| replayed:", final.journal[3].output["replayed"])

retry refused: run_531eddcd76e4 leased by worker-1 until 1700000060
run run_531eddcd76e4  status=SUCCEEDED  version=13  steps=5  tokens=1060  cost=$0.0009
  [0] llm    done    decide             key=-                    {"kind": "tool_call", "tool": "lookup_price", "args": {"sku": "ABC"}, 
  [1] tool   done    lookup_price       key=run_531eddcd76e4:1   {"result": {"sku": "ABC", "price": 42.0}, "replayed": false}
  [2] llm    done    decide             key=-                    {"kind": "tool_call", "tool": "charge_card", "args": {"amount": 42.0},
  [3] tool   done    charge_card        key=run_531eddcd76e4:3   {"result": {"charge_id": "run_531eddcd76e4:3", "amount": 42.0, "replay
  [4] llm    done    decide             key=-                    {"kind": "final", "tool": null, "args": {}, "text": "Charged 42.00 for
charges: 1 | recoveries: 1 | replayed: True


One charge. The retry found the memoised result (`replayed=True`), finished the journal entry, and the loop went on to the final answer.

**Why record intent first?** Because the model is non-deterministic. If the retry had *re-asked the LLM*, it might have said `charge_card(amount=42.5)` — a different key, a second charge. Journaling the decision turns a probabilistic step into a replayable one.

## 3. Duplicate delivery (at-least-once)

In [6]:
gateway3 = PaymentGateway()
loop, dispatcher, clock = build(gateway=gateway3)
tools.get("charge_card").fn = lambda a, c: gateway3.charge(float(a["amount"]), idempotency_key=c.idempotency_key)
run = loop.start("Buy SKU ABC")
dispatcher.deliver_one(loop.handle)
dispatcher.duplicate_next()                          # Cloud Tasks hands us the same task twice
dispatcher.drain(loop.handle)
final = loop.store.get(run.run_id)
print("status:", final.status.value, "| journal:", len(final.journal), "| charges:", len(gateway3.charges))
print("task names seen (de-dup keys):", sorted(dispatcher.seen_names))

status: SUCCEEDED | journal: 5 | charges: 1
task names seen (de-dup keys): ['run_a1b4985ddf8a-step-0', 'run_a1b4985ddf8a-step-2', 'run_a1b4985ddf8a-step-4']


The second delivery finds the step already done (`next_index > expected_index`), re-enqueues the *named* next task (collapsed by de-dup) and returns 200.

## 4. Budget = the real stop condition

In [7]:
endless = [Decision.call("lookup_price", sku="X")] * 100
loop, dispatcher, clock = build(script=endless)
run = loop.start("loop forever", budget=Budget(max_steps=6, max_cost_usd=0.01))
dispatcher.drain(loop.handle)
final = loop.store.get(run.run_id)
print(final.status.value, "|", final.error, "| tokens:", final.usage.tokens, f"| cost: ${final.usage.cost_usd:.5f}")

FAILED | budget: max_steps 6 reached | tokens: 1020 | cost: $0.00081


## 5. The counter-example
Same crash, but the downstream API has **no idempotency key** and we bypass the memo store.

In [8]:
naive = NaivePaymentGateway()
class NoMemo(InMemoryIdempotencyStore):
    def __contains__(self, k): return False
faults = FaultInjector(); clock = FakeClock(); dispatcher = InMemoryDispatcher(clock=clock)
loop = DurableAgentLoop(store=InMemoryRunStore(clock=clock), llm=ScriptedLLM(SCRIPT[1:]),
                        tools=ToolRegistry([Tool("charge_card", "charge", lambda a, c: naive.charge(float(a["amount"])))]),
                        dispatcher=dispatcher, idempotency=NoMemo(), clock=clock, faults=faults)
run = loop.start("charge")
try:
    loop.handle(dispatcher.queue.pop(0))
except SimulatedCrash:
    pass
clock.advance(61); loop.step(run.run_id, expected_index=2)
print("charges with a naive gateway after one crash:", naive.charges)

charges with a naive gateway after one crash: [42.0]


## 6. Async tools: park, don't poll in-process
A tool that kicks off a 2-minute job returns a ticket. The run parks (`WAITING_EVENT`) and schedules a **delayed** Cloud Task to poll with exponential back-off — or an external webhook resumes it. No process waits.

In [9]:
clock = FakeClock()
jobs = SlowJobService(clock, duration_s=120)
atools = ToolRegistry([Tool("run_report", "start a long report", lambda a, c: jobs.submit(a, idempotency_key=c.idempotency_key),
                            is_async=True, poll=jobs.status)])
dispatcher = InMemoryDispatcher(clock=clock)
loop = DurableAgentLoop(store=InMemoryRunStore(clock=clock), llm=ScriptedLLM([Decision.call("run_report", region="apac"),
                        lambda msgs: Decision.final(msgs[-1]["content"])]), tools=atools, dispatcher=dispatcher,
                        idempotency=InMemoryIdempotencyStore(), clock=clock, poll_base_s=10, poll_max_s=60)
run = loop.start("report")
dispatcher.drain(loop.handle)
while loop.store.get(run.run_id).status == RunStatus.WAITING_EVENT:
    nxt = dispatcher.queue[0]
    print(f"t={clock()-1_700_000_000:>4.0f}s  parked; next poll in {nxt.not_before-clock():.0f}s  (task {nxt.name})")
    clock.t = nxt.not_before; dispatcher.drain(loop.handle)
show_journal(loop.store.get(run.run_id))

t=   0s  parked; next poll in 10s  (task run_99a2dad1bd98-poll-1-0)
t=  10s  parked; next poll in 20s  (task run_99a2dad1bd98-poll-1-1)
t=  30s  parked; next poll in 40s  (task run_99a2dad1bd98-poll-1-2)
t=  70s  parked; next poll in 60s  (task run_99a2dad1bd98-poll-1-3)
run run_99a2dad1bd98  status=SUCCEEDED  version=20  steps=3  tokens=720  cost=$0.0000
  [0] llm    done    decide             key=-                    {"kind": "tool_call", "tool": "run_report", "args": {"region": "apac"}
  [1] tool   done    run_report         key=run_99a2dad1bd98:1   {"ticket": "job_run_99a2dad1bd98:1", "pending": true, "replayed": fals
  [2] event  done    run_report:result  key=-                    {"result": {"summary": "processed {'region': 'apac'}"}}
  [3] llm    done    decide             key=-                    {"kind": "final", "tool": null, "args": {}, "text": "EVENT[run_report:


## 7. Mapping to GCP
| In this notebook | On GCP | File |
|---|---|---|
| `InMemoryDispatcher` | Cloud Tasks queue, HTTP target with OIDC token, task **name** = `run-step-N` | `core/transport.py` |
| `InMemoryRunStore` | Firestore document per run, transactions for `version` + `lease` | `core/firestore_store.py` |
| `InMemoryIdempotencyStore` | Firestore `idempotency_keys` collection with TTL | `core/firestore_store.py` |
| `ScriptedLLM` | Gemini on Vertex AI via `google-genai` | `core/llm.py` |
| `loop.handle(env)` | `POST /internal/tasks/step` on Cloud Run | `service/app.py` |
| `FakeClock.advance` | lease TTL / `schedule_time` on tasks | — |

In [10]:
from lragents.core.transport import Envelope
for e in [Envelope("run_ab12", 3, "step"), Envelope("run_ab12", 3, "step"), Envelope("run_ab12", 1, "poll", payload={"attempt": 2})]:
    print(e.name)
import inspect; from lragents.service import app as service
print(inspect.getsource(service.create_app).split('@app.post("/internal/tasks/{kind}"')[1][:700])

run_ab12-step-3
run_ab12-step-3
run_ab12-poll-1-2


, dependencies=[Depends(verify_oidc)])
    def task(kind: str, env: dict, request: Request):
        """Return 2xx only when the step is durably done; any exception → 5xx → Cloud Tasks retries."""
        envelope = Envelope(**env)
        try:
            run = svc.loop.handle(envelope)
        except LeaseHeld as e:                                 # another worker is on it: ask for a retry later
            raise HTTPException(status_code=429, detail=str(e))
        return {"run_id": run.run_id, "status": run.status, "next_index": run.next_index(),
                "retry_count": request.headers.get("X-CloudTasks-TaskRetryCount")}

    @app.post("/internal/pubsub/push", dependencies=[Depend


## Takeaways
1. **One step per wake-up**; the store is the only memory.
2. **Intent before side effect** — the LLM is a non-deterministic side effect; journal its decision, then act.
3. **Idempotency key = `run:step`**, passed downstream *and* memoised locally.
4. **Leases expire**; a retry that arrives too early is refused, not raced.
5. **Budgets are code**, not prompt text.
6. **Waiting is free**: async tools park the run; polls are delayed tasks with back-off.